In [3]:
import os
import shutil
import time
from datetime import datetime, timedelta

# مسیر مبدا و مقصد
source_dir = r"D:\PT\edge_rl_3"
dest_dir = r"C:\Users\Admin\Desktop\edge_rl_3"

# محاسبه زمان ۳ ساعت قبل
cutoff_time = datetime.now() - timedelta(hours=3)

# تبدیل به timestamp برای مقایسه با mtime
cutoff_timestamp = cutoff_time.timestamp()

# پیمایش دایرکتوری مبدا
for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.endswith(".py"):
            file_path = os.path.join(root, file)
            
            # دریافت زمان آخرین ویرایش
            mtime = os.path.getmtime(file_path)
            
            # اگر فایل در ۳ ساعت اخیر ویرایش شده
            if mtime >= cutoff_timestamp:
                # محاسبه مسیر نسبی نسبت به مبدا
                rel_path = os.path.relpath(root, source_dir)
                dest_subdir = os.path.join(dest_dir, rel_path)
                
                # ساخت پوشه مقصد در صورت نیاز
                os.makedirs(dest_subdir, exist_ok=True)
                
                # مسیر کامل فایل مقصد
                dest_file = os.path.join(dest_subdir, file)
                
                # کپی فایل
                shutil.copy2(file_path, dest_file)
                print(f"کپی شد: {file_path} -> {dest_file}")

کپی شد: D:\PT\edge_rl_3\algorithms\base.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\base.py
کپی شد: D:\PT\edge_rl_3\algorithms\greedy\greedy_algorithm.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\greedy\greedy_algorithm.py
کپی شد: D:\PT\edge_rl_3\algorithms\hpa\hpa_algorithm.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\hpa\hpa_algorithm.py
کپی شد: D:\PT\edge_rl_3\algorithms\ppo\env.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\ppo\env.py
کپی شد: D:\PT\edge_rl_3\algorithms\ppo\ppo_algorithm.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\ppo\ppo_algorithm.py
کپی شد: D:\PT\edge_rl_3\algorithms\voila\voila_algorithm.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\voila\voila_algorithm.py
کپی شد: D:\PT\edge_rl_3\common\metrics.py -> C:\Users\Admin\Desktop\edge_rl_3\common\metrics.py
کپی شد: D:\PT\edge_rl_3\common\models.py -> C:\Users\Admin\Desktop\edge_rl_3\common\models.py
کپی شد: D:\PT\edge_rl_3\k8s_adapter\realtime_dispatcher.py -> C:\Users\Admin\Desktop\edge_rl

In [14]:
#کپی فایلهای اصلی پروژه
import os
import shutil
import fnmatch
import stat
import zipfile
from pathlib import Path
from datetime import datetime

def force_remove_readonly(func, path, excinfo):
    """
    تابع کمکی برای حذف فایل‌های فقط خواندنی
    """
    os.chmod(path, stat.S_IWRITE)
    func(path)

def clear_directory(directory):
    """
    پاک کردن کامل محتویات یک دایرکتوری با مدیریت خطاها
    """
    dir_path = Path(directory)
    
    if not dir_path.exists():
        #print(f"📁 پوشه {directory} وجود ندارد، ایجاد می‌شود...")
        dir_path.mkdir(parents=True, exist_ok=True)
        return True
    
    #print(f"🧹 پاک کردن محتویات: {directory}")
    
    success = True
    for item in dir_path.iterdir():
        try:
            if item.is_file() or item.is_symlink():
                try:
                    item.chmod(stat.S_IWRITE)
                except:
                    pass
                item.unlink()
                #print(f"  🗑️ حذف فایل: {item.name}")
                
            elif item.is_dir():
                try:
                    shutil.rmtree(item, onerror=force_remove_readonly)
                    #print(f"  🗑️ حذف پوشه: {item.name}")
                except Exception as e:
                    #print(f"  ⚠️ خطا در حذف پوشه {item.name}: {e}")
                    try:
                        for root, dirs, files in os.walk(item):
                            for d in dirs:
                                try:
                                    os.chmod(os.path.join(root, d), stat.S_IWRITE)
                                except:
                                    pass
                            for f in files:
                                try:
                                    os.chmod(os.path.join(root, f), stat.S_IWRITE)
                                except:
                                    pass
                        shutil.rmtree(item)
                        #print(f"  🗑️ حذف پوشه (تلاش دوم): {item.name}")
                    except Exception as e2:
                        #print(f"  ❌ خطا در حذف پوشه {item.name}: {e2}")
                        success = False
                        
        except Exception as e:
            #print(f"  ❌ خطا در حذف {item.name}: {e}")
            success = False
    
    return success

def create_zip(source_dir, zip_name=None):
    """
    ایجاد فایل ZIP از محتویات یک پوشه
    """
    source_path = Path(source_dir)
    
    if not source_path.exists():
        #print(f"❌ خطا: پوشه {source_dir} وجود ندارد!")
        return None
    
    # اگر نام ZIP مشخص نشده، از نام پوشه استفاده کن
    if zip_name is None:
        zip_name = f"{source_path.name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
    else:
        # اطمینان از پسوند .zip
        if not zip_name.endswith('.zip'):
            zip_name += '.zip'
    
    zip_path = source_path / zip_name
    
    #print(f"\n📦 ایجاد فایل ZIP: {zip_path}")
    
    try:
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, dirs, files in os.walk(source_path):
                for file in files:
                    file_path = Path(root) / file
                    # محاسبه مسیر نسبی برای ذخیره در ZIP
                    arcname = file_path.relative_to(source_path)
                    
                    # از ZIP کردن خود فایل ZIP جلوگیری کن
                    if str(arcname) == zip_name:
                        continue
                    
                    zipf.write(file_path, arcname)
                    #print(f"  📄 اضافه شد به ZIP: {arcname}")
        
        #print(f"✅ فایل ZIP با موفقیت ایجاد شد: {zip_path}")
        #print(f"📊 حجم فایل ZIP: {zip_path.stat().st_size / 1024:.2f} KB")
        return zip_path
        
    except Exception as e:
        #print(f"❌ خطا در ایجاد ZIP: {e}")
        return None

def copy_with_gitignore(source_dir, dest_dir, gitignore_path=None, create_zip_file=True):
    """
    کپی فایل‌ها و پوشه‌ها از source_dir به dest_dir با رعایت .gitignore
    و نادیده گرفتن پوشه‌های .git، output، log
    و سپس ایجاد فایل ZIP از محتویات مقصد
    """
    
    # تبدیل به Path object برای سهولت کار
    source_path = Path(source_dir)
    dest_path = Path(dest_dir)
    
    # بررسی وجود منبع
    if not source_path.exists():
        #print(f"❌ خطا: مسیر منبع {source_dir} وجود ندارد!")
        return 0, 0, None
    
    # اگر فایل .gitignore در مسیر مشخص نشده، در source_dir جستجو کن
    if gitignore_path is None:
        gitignore_path = source_path / '.gitignore'
    else:
        gitignore_path = Path(gitignore_path)
    
    # خواندن الگوهای .gitignore
    ignore_patterns = []
    if gitignore_path.exists():
        #print(f"📄 خواندن .gitignore از: {gitignore_path}")
        with open(gitignore_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#'):
                    ignore_patterns.append(line)
    #else:
        #print(f"ℹ️ فایل .gitignore در {gitignore_path} یافت نشد")
    
    # اضافه کردن پوشه‌های مورد نظر به لیست الگوهای نادیده گرفته شده
    ignore_patterns.extend([
        '.git',
        '.git/',
        'outputs',
        'outputs/',
        'logs',
        'logs/',
        'test.ipynb',
        '.gitignore'
    ])
    
    
    # تابع بررسی اینکه آیا فایل/پوشه باید نادیده گرفته شود
    def should_ignore(path):
        try:
            rel_path = path.relative_to(source_path)
        except ValueError:
            return True
            
        rel_str = str(rel_path).replace('\\', '/')
        
        # بررسی ویژه برای پوشه‌های خاص
        if (rel_str == '.git' or rel_str.startswith('.git/') or 
            rel_str == 'output' or rel_str.startswith('output/') or
            rel_str == 'log' or rel_str.startswith('log/')):
            return True
        
        # بررسی الگوهای gitignore
        for pattern in ignore_patterns:
            if pattern.endswith('/'):
                if rel_str.startswith(pattern) or rel_str == pattern[:-1]:
                    return True
            else:
                if '*' in pattern or '?' in pattern:
                    if fnmatch.fnmatch(rel_str, pattern):
                        return True
                    parts = rel_str.split('/')
                    for i in range(len(parts)):
                        partial = '/'.join(parts[:i+1])
                        if fnmatch.fnmatch(partial, pattern):
                            return True
                else:
                    if rel_str == pattern or rel_str.startswith(pattern + '/'):
                        return True
        return False
    
    # پاک کردن محتویات دایرکتوری مقصد
    if not clear_directory(dest_path):
        print("⚠️ برخی موارد در پاک کردن مقصد با مشکل مواجه شدند، اما ادامه می‌دهیم...")
    
    # اطمینان از وجود دایرکتوری مقصد
    dest_path.mkdir(parents=True, exist_ok=True)
    
    # کپی فایل‌ها و پوشه‌ها
    #print(f"\n📁 شروع کپی از {source_path} به {dest_path}")
    #print(f"🚫 نادیده گرفتن پوشه‌های: .git, output, log\n")
    
    copied_count = 0
    ignored_count = 0
    
    for root, dirs, files in os.walk(source_path):
        root_path = Path(root)
        
        # فیلتر کردن پوشه‌ها
        dirs_to_remove = []
        for d in dirs:
            dir_path = root_path / d
            if should_ignore(dir_path):
                dirs_to_remove.append(d)
                ignored_count += 1
                #print(f"⏭️ نادیده گرفته شد: {dir_path.relative_to(source_path)}/")
        
        for d in dirs_to_remove:
            dirs.remove(d)
        
        # کپی فایل‌ها
        for file in files:
            source_file = root_path / file
            try:
                rel_path = source_file.relative_to(source_path)
            except ValueError:
                continue
            
            if should_ignore(source_file):
                ignored_count += 1
                #print(f"⏭️ نادیده گرفته شد: {rel_path}")
                continue
            
            dest_file = dest_path / rel_path
            dest_file.parent.mkdir(parents=True, exist_ok=True)
            
            try:
                shutil.copy2(source_file, dest_file)
                copied_count += 1
                #print(f"✅ کپی شد: {rel_path}")
            except Exception as e:
                print(f"❌ خطا در کپی {rel_path}: {e}")
    
    # ایجاد فایل ZIP
    zip_file = None
    if create_zip_file and copied_count > 0:
        zip_file = create_zip(dest_path, f"{dest_path.name}.zip")
    
    return copied_count, ignored_count, zip_file

def copy_simple_with_zip(source_dir, dest_dir):
    """
    نسخه ساده - فقط کپی و ZIP
    """
    source_path = Path(source_dir)
    dest_path = Path(dest_dir)
    
    if not source_path.exists():
        #print(f"❌ خطا: مسیر منبع {source_dir} وجود ندارد!")
        return None
    
    # پاک کردن مقصد
    if not clear_directory(dest_path):
        #print("⚠️ خطا در پاک کردن مقصد!")
        return None
    
    # کپی با نادیده گرفتن پوشه‌های مشخص
    #print(f"\n📁 کپی ساده از {source_dir} به {dest_dir}")
    try:
        shutil.copytree(
            source_dir, 
            dest_dir, 
            ignore=shutil.ignore_patterns('.git', 'output', 'log'),
            dirs_exist_ok=True
        )
        #print("✅ کپی با موفقیت انجام شد!")
        
        # ایجاد ZIP
        zip_file = create_zip(dest_path, f"{dest_path.name}.zip")
        return zip_file
        
    except Exception as e:
        #print(f"❌ خطا در کپی: {e}")
        return None

# استفاده از تابع
if __name__ == "__main__":
    source = r"D:\PT\edge_rl_3"
    destination = r"C:\Users\Admin\Desktop\edge_rl_3"
    
    #print("=" * 70)
    #print("📋 اسکریپت کپی با رعایت .gitignore و ایجاد ZIP")
    #print("=" * 70)
    #print(f"📂 منبع: {source}")
    #print(f"📂 مقصد: {destination}")
    #print("=" * 70 + "\n")
    
    try:
        # روش کامل با .gitignore و ZIP
        #print("🔄 در حال اجرا...")
        copied, ignored, zip_file = copy_with_gitignore(source, destination, create_zip_file=True)
        
        #print("\n" + "=" * 70)
        #print("📊 گزارش نهایی:")
        print(f"✅ تعداد فایل‌های کپی شده: {copied}")
        #print(f"🚫 تعداد موارد نادیده گرفته شده: {ignored}")
        #print(f"🚫 پوشه‌های نادیده گرفته شده: .git, output, log")
        #if zip_file:
            #print(f"📦 فایل ZIP ایجاد شده: {zip_file}")
            #print(f"📊 مسیر ZIP: {zip_file.parent / zip_file.name}")
        #print("=" * 70)
        
    except Exception as e:
        #print(f"\n❌ خطا در روش کامل: {e}")
        #print("\n🔄 تلاش با روش ساده...")
        
        try:
            zip_file = copy_simple_with_zip(source, destination)
            #if zip_file:
                #print(f"\n✅ فایل ZIP ایجاد شد: {zip_file}")
        except Exception as e2:
            print(f"❌ خطا در روش ساده: {e2}")

✅ تعداد فایل‌های کپی شده: 47


In [2]:
# verify_all_fixes.py
"""بررسی وجود همه‌ی فیکس‌های این گفتگو، قبل از شروع آموزش نهایی PPO."""
import inspect

checks = []

def check(name, condition):
    status = "✓" if condition else "✗ MISSING"
    checks.append((name, condition))
    print(f"[{status}] {name}")

# --- دور ۱: مدل دامنه و متریک ---
from common.models import Server
src = inspect.getsource(Server.instantaneous_utilization)
check("۱.۱ DRAINING در utilization", "DRAINING" in src)

from common.metrics import MetricsCollector
src = inspect.getsource(MetricsCollector.record_snapshot)
check("۱.۲ load_balance_cv با ۱ سرور فعال", "len(active) == 1" in src)

# --- دور ۲: capacity-starved ---
from algorithms.base import AlgorithmBase
src = inspect.getsource(AlgorithmBase._capacity_starved_services)
check("۲.۱ capacity_starved شامل BOOTING", "BOOTING" in src)
check("۲.۱ threshold پارامتری شده", "occ_threshold" in src)

from algorithms.voila.voila_algorithm import VoilaAlgorithm
src = inspect.getsource(VoilaAlgorithm.provision_decision)
check("۲.۲ Voila با OCC_UP_THRESHOLD خودش", "OCC_UP_THRESHOLD" in src and "_capacity_starved_services" in src)

from simulator.engine import SimulationEngine
src = inspect.getsource(SimulationEngine._any_service_capacity_starved)
check("۲.۳ engine._any_service_capacity_starved شامل BOOTING", "BOOTING" in src)

# --- دور ۳: n_ready_replicas در snapshot ---
src = inspect.getsource(SimulationEngine._build_metrics_snapshot)
check("۳ n_ready_replicas در snapshot معمولی", '"n_ready_replicas"' in src)
src_ro = inspect.getsource(SimulationEngine._build_metrics_snapshot_readonly)
check("۳ n_ready_replicas در snapshot readonly", '"n_ready_replicas"' in src_ro)

# --- دور ۴: action mask ---
from algorithms.ppo.env import EdgeResourceEnv
src = inspect.getsource(EdgeResourceEnv._any_server_can_host)
check("۴.۱ mask can_up چک ACTIVE", "ServerState.ACTIVE" in src)
src = inspect.getsource(EdgeResourceEnv.action_masks)
check("۴.۱ mask can_down از n_ready/mature_ready", "ready_replicas" in src.lower() or "n_ready_replicas" in src or "n_mature_ready_replicas" in src)

from algorithms.ppo.ppo_algorithm import PPOAlgorithm
src = inspect.getsource(PPOAlgorithm._build_action_masks)
check("۴.۲ ppo_algorithm mask چک ACTIVE", "ServerState.ACTIVE" in src)

# --- دور ۵: demand_centroid در لحظه‌ی ورود ---
src = inspect.getsource(SimulationEngine._handle_arrival)
n_appends = src.count("_recent_positions[req.service_id].append")
check("۵ demand_centroid فقط یک‌بار append میشه (نه دوبار)", n_appends == 1)

# --- دور ۶: ENERGY_RESYNC + drain دینامیک ---
from simulator.events import EventType
check("۶.۱ ENERGY_RESYNC در EventType", hasattr(EventType, "ENERGY_RESYNC"))
check("۶.۲ ENERGY_RESYNC در step()", "ENERGY_RESYNC" in inspect.getsource(SimulationEngine.step))
src = inspect.getsource(SimulationEngine._start_replica_drain)
check("۶.۳ drain دینامیک (available_at)", "available_at" in src)

# --- دور بعد: خودارجاعی TURN_ON/TURN_OFF ---
check("۷.۱ _was_turn_on_necessary_audit موجود", hasattr(SimulationEngine, "_was_turn_on_necessary_audit"))
check("۷.۲ _was_turn_off_necessary_audit موجود", hasattr(SimulationEngine, "_was_turn_off_necessary_audit"))

# --- proximity در فاز ۳ ---
try:
    from k8s_adapter.realtime_dispatcher import RealtimeEngine
    src = inspect.getsource(RealtimeEngine.__init__)
    check("۸ _tick_proximity_violated در realtime_dispatcher", "_tick_proximity_violated" in src)
except ImportError:
    print("[SKIP] k8s_adapter نیاز به کتابخانه‌ی kubernetes/redis دارد - چک نشد")

# --- bias سرور اول ---
src = inspect.getsource(EdgeResourceEnv.step)
check("۹.۱ رفع bias در env.py (نه break ساده)", "np_random.choice" in src or "random" in src.lower())
src = inspect.getsource(PPOAlgorithm._predict_and_cache)
check("۹.۲ رفع bias در ppo_algorithm.py", "_tie_break_rng" in src or "random" in src.lower())

# --- محافظت سن replica ---
src = inspect.getsource(SimulationEngine._apply_scale_decision)
check("۱۰ محافظت سن replica (created_at/mature)", "created_at" in src or "mature" in src)

# --- کالیبراسیون ---
from common import state_builder
check("۱۱.۱ _NORM_RESPONSE_TIME_SEC بازکالیبره", abs(state_builder._NORM_RESPONSE_TIME_SEC - 300.0) > 1)
check("۱۱.۲ _NORM_ENERGY_JOULE بازکالیبره", abs(state_builder._NORM_ENERGY_JOULE - 12000.0) > 1)
check("۱۱.۳ _NORM_ARRIVAL_RATE بازکالیبره", abs(state_builder._NORM_ARRIVAL_RATE - 20.0) > 1)

import algorithms.ppo.env as env_mod
check("۱۱.۴ _NORM_REJECTED_PER_TICK بازکالیبره یا تأییدشده", env_mod._NORM_REJECTED_PER_TICK == 6.0)

# --- خلاصه ---
print("\n" + "="*50)
total = len(checks)
passed = sum(1 for _, ok in checks if ok)
print(f"نتیجه: {passed}/{total} فیکس تأیید شد")
if passed < total:
    print("⚠️ قبل از آموزش، موارد ✗ بالا را برطرف کن!")
else:
    print("✅ همه‌ی فیکس‌ها موجودند — آماده‌ی آموزش نهایی")

[✗ MISSING] ۱.۱ DRAINING در utilization
[✓] ۱.۲ load_balance_cv با ۱ سرور فعال
[✓] ۲.۱ capacity_starved شامل BOOTING
[✓] ۲.۱ threshold پارامتری شده
[✓] ۲.۲ Voila با OCC_UP_THRESHOLD خودش
[✓] ۲.۳ engine._any_service_capacity_starved شامل BOOTING
[✓] ۳ n_ready_replicas در snapshot معمولی
[✓] ۳ n_ready_replicas در snapshot readonly
[✓] ۴.۱ mask can_up چک ACTIVE
[✓] ۴.۱ mask can_down از n_ready/mature_ready
[✓] ۴.۲ ppo_algorithm mask چک ACTIVE
[✓] ۵ demand_centroid فقط یک‌بار append میشه (نه دوبار)
[✓] ۶.۱ ENERGY_RESYNC در EventType
[✓] ۶.۲ ENERGY_RESYNC در step()
[✓] ۶.۳ drain دینامیک (available_at)
[✓] ۷.۱ _was_turn_on_necessary_audit موجود
[✓] ۷.۲ _was_turn_off_necessary_audit موجود
[SKIP] k8s_adapter نیاز به کتابخانه‌ی kubernetes/redis دارد - چک نشد
[✓] ۹.۱ رفع bias در env.py (نه break ساده)
[✓] ۹.۲ رفع bias در ppo_algorithm.py
[✓] ۱۰ محافظت سن replica (created_at/mature)
[✓] ۱۱.۱ _NORM_RESPONSE_TIME_SEC بازکالیبره
[✓] ۱۱.۲ _NORM_ENERGY_JOULE بازکالیبره
[✓] ۱۱.۳ _NORM_ARRIVAL_RATE بازک

In [ ]:
# 1. دوباره آموزش
EOTCH_SEED=42 python3 -m algorithms.ppo.train

# 2. ارزیابی
python3 -m evaluation.compare_runs --output-dir outputs/test_v2

# 3. مقایسه
python3 -m evaluation.aggregate_seeds --seeds 42 --base-dir outputs/test_v2